# Airport Operations Analytics — Pipeline Walkthrough**MSBA 305 — Data Processing Framework | Spring 2025/2026****Author:** Mohamad HassanThis notebook walks through the full data pipeline from ingestion to analytical queries and visualisation. Each stage is annotated and each intermediate result is displayed inline.---## Contents1. [Environment setup](#1)2. [Ingestion — three data sources](#2)3. [Cleaning and transformation](#3)4. [Integration — merged analytical dataset](#4)5. [Analytical queries (Q1 – Q5)](#5)6. [Visualisations](#6)7. [Summary and next steps](#7)

## <a id='1'></a>1. Environment setupAll secrets are loaded from environment variables — no credentials hardcoded in this notebook.

In [ ]:
import osimport sysimport loggingimport pandas as pdimport matplotlib.pyplot as pltimport requests# Surface the pipeline module (one directory up)sys.path.insert(0, "../code")import pipelinelogging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")pd.set_option("display.max_columns", 20)pd.set_option("display.width", 120)print("pandas:", pd.__version__)print("Pipeline module loaded from:", pipeline.__file__)

## <a id='2'></a>2. Ingestion — three data sourcesThree distinct sources, each with their own ingestion function in `pipeline.py`. Errors are caught and surfaced with context; the API ingestion degrades gracefully to `None` on failure.

### 2.1 BTS Flight Delay Dataset (CSV)

In [ ]:
# For a live demo, use the real BTS CSV; here we use the structural sample shipped in data/FLIGHT_DATA_PATH = os.getenv("FLIGHT_DATA_PATH", "../data/bts_sample.csv")flights_raw = pd.read_csv(FLIGHT_DATA_PATH)print(f"Rows: {len(flights_raw):,}  |  Columns: {flights_raw.shape[1]}")flights_raw.head()

### 2.2 OpenFlights Airport Metadata (CSV via URL)

In [ ]:
airports = pipeline.ingest_airports(pipeline.OPENFLIGHTS_URL)print(f"Rows: {len(airports):,}  (after filtering invalid IATA codes)")airports.head()

### 2.3 AviationStack API (real-time JSON)Returns `None` if the API key is missing or the endpoint is unreachable — the pipeline continues without real-time data rather than failing.

In [ ]:
api_df = pipeline.ingest_realtime_flights(os.getenv("AVIATIONSTACK_API_KEY", ""))if api_df is None:    print("API skipped (no key set or endpoint unavailable) — continuing on historical path.")else:    print(f"Live records received: {len(api_df):,}")    api_df.head()

## <a id='3'></a>3. Cleaning and transformationPer-column missing-value strategy:- `arr_delay`, `arr_flights` → **delete rows** with nulls (imputation would distort totals).- `city`, `country` → delete rows after merge (preserves geographic interpretability).- API `delay` field → filter; free-tier returns ~90% nulls here.A derived `date` field is built from `year` + `month` to support time-series grouping.

In [ ]:
flights_clean = pipeline.clean_flight_delays(flights_raw)print(f"Rows before: {len(flights_raw):,}   After: {len(flights_clean):,}")flights_clean.head()

## <a id='4'></a>4. Integration — merged analytical datasetTwo-step integration:1. BTS ⟵ OpenFlights on `airport = iata`, adding city/country context.2. Result ⟵ AviationStack on arrival airport, optional enrichment.The result — `merged` — is the query-ready analytical dataset referenced throughout §A7 of the report.

In [ ]:
merged = pipeline.integrate_sources(flights_clean, airports, api_df)print(f"Integrated dataset: {len(merged):,} rows × {merged.shape[1]} columns")print(f"Null rate in core fields: {merged[['arr_flights','arr_delay','country']].isna().mean().to_dict()}")merged.head()

## <a id='5'></a>5. Analytical queries (Q1 – Q5)Five queries of increasing complexity answer the KPIs defined in §A1 of the report. Each is a pure function in `pipeline.py` that returns a pandas object.

### Q1 — Filter: high-delay airport-months*Business question: Which airport-months represent severe congestion episodes worth investigating?*

In [ ]:
pipeline.query_1_filter_high_delay_months(merged).head(10)

### Q2 — Aggregation: flights by country*Business question: Where is air traffic concentrated?*

In [ ]:
pipeline.query_2_aggregation_country_volume(merged).head()

### Q3 — Join + aggregation: top 10 airports enriched*Business question: Which airports are the highest-volume hubs? What is their delay-per-flight?*

In [ ]:
pipeline.query_3_join_top_airports_enriched(merged)

### Q4 — Window function: rolling 3-month delay (ATL)*Business question: Is this airport's delay performance trending up or down?*

In [ ]:
rolling = pipeline.query_4_window_rolling_delay(merged, airport_code="ATL")rolling.tail(12)

### Q5 — Multi-dimensional pivot: delay-type decomposition*Business question: For each top airport, which delay cause dominates — and therefore which intervention family is justified?*

In [ ]:
pipeline.query_5_pivot_delay_decomposition(merged)

## <a id='6'></a>6. VisualisationsThe same four matplotlib charts discussed in §A8 of the report. Each is rendered directly from the query outputs above.

In [ ]:
pipeline.plot_top_countries(pipeline.query_2_aggregation_country_volume(merged))

In [ ]:
pipeline.plot_top_countries(pipeline.query_2_aggregation_country_volume(merged), exclude_us=True)

In [ ]:
pipeline.plot_top_airports(pipeline.query_3_join_top_airports_enriched(merged))

In [ ]:
pipeline.plot_delay_types(merged)

## <a id='7'></a>7. Summary and next steps### Key findings from this run- **Late-aircraft delay dominates** — typically >50% of total delay minutes at top hubs (see Q5 and Chart 4). This is the single most actionable signal in the analysis.- **Volume is concentrated** — ~10 airports carry the majority of arrivals (Q3 / Chart 3).- **Performance varies across peers** — mean delay per flight is not uniform across the top-10 cohort (Q3), supporting peer-benchmarking.### Strategic recommendations (see report §A12.1)- **R1** — Prioritise turnaround and recovery protocols at top-5 hubs (drives down propagation).- **R2** — Schedule-buffer tuning informed by Q4 rolling signal.- **R3** — Peer-benchmarking cycle across top-10 cohort.- **R4** — Weather API integration for causal corroboration.- **R5** — Eurocontrol + ICAO integration to close the geographic gap.### Next phases (see report §A12.2)- Phase 2 (0–6 months): PostgreSQL storage, cloud object store, scheduled orchestration, 6+ sources, daily refresh.- Phase 3 (6–18 months): cloud data warehouse, streaming AviationStack, predictive forecasting layer, executive dashboards.